# Deal Prioritizer: API walkthrough (live on Render)

The same walkthrough, run against the deployed API. Nothing to start locally; the only dependency is `httpx` (`pip install httpx`).

- App: https://deal-prioritizer-web.onrender.com
- API: https://deal-prioritizer-api.onrender.com
- Swagger: https://deal-prioritizer-api.onrender.com/docs

The API is on Render's free plan, so after it's been idle the first request can take about a minute while it wakes up.

One difference from running locally: overpass-api.de blocks connections from some cloud IP ranges, Render's included, so in production the listings come from Nominatim's search for the same OSM tags. `source_detail` in step 1 says when that happened. It's the same OpenStreetMap data, just fewer results per search.

For running it on your own machine, see [`api_walkthrough_local.ipynb`](api_walkthrough_local.ipynb).

Business data © OpenStreetMap contributors, ODbL.

In [1]:
import csv
import io
from collections import Counter

import httpx

API = "https://deal-prioritizer-api.onrender.com"
client = httpx.Client(base_url=API, timeout=120)
print(client.get("/healthz").json())

{'ok': True, 'app': 'deal-prioritizer'}


## 1. Run a search

One request does the whole thing: geocode the market, pull matching businesses from OpenStreetMap, dedupe, score, and save the run in Postgres. If you run the same search again within the hour, it comes from the cache.

In [2]:
res = client.post(
    "/v1/pipeline/runs",
    json={"vertical": "Auto", "market": "Chicago, IL", "limit": 35},
)
run = res.json()
print(res.status_code, "| source:", run["source_status"], "|", len(run["targets"]), "companies")
print("detail:", run["source_detail"] or "none")

201 | source: live | 30 companies
detail: Overpass: All connection attempts failed; used Nominatim search instead


## 2. The ranked shortlist

Every score comes with the reasons behind it, so whoever makes the calls can see why a shop ranks where it does.

In [3]:
shortlist = [t for t in run["targets"] if not t["skipped"]][:10]
for t in shortlist:
    contact = t["main_phone"] or t["email"] or "-"
    print(f"{t['fit_score']:>5} {t['tier']:<6} {t['legal_name'][:32]:<33} {contact:<16} {t['rationale']}")

 80.0 prime  Grand Auto Center                 +1-312-226-1500  Phone listed · Street address · Hours listed · Website listed
 80.0 prime  Augusta & Paulina Auto Service    +1-773-486-9768  Phone listed · Street address · Hours listed · Website listed
 80.0 prime  Gateway Auto Service              +1-773-342-7105  Phone listed · Street address · Hours listed · Website listed
 75.0 solid  Erie LaSalle Body Shop            +1-312-337-3903  Phone listed · Street address · Website listed
 70.0 solid  JC Garage                         +1 872 342 3850  Phone listed · Street address
 70.0 solid  Arandas Tires & Rims              +1-773-252-6292  Phone listed · Street address
 63.0 solid  Fulton-Desplaines Garage          -                Street address · Website listed
 58.0 solid  Tony's Auto Center                -                Street address · No contact details in OSM, enrich first
 58.0 solid  Ogden Automotive                  -                Street address · No contact details in OSM

## 3. Chains get filtered out

OpenStreetMap tags franchise locations with `brand:wikidata`, which is enough to reject the national chains before anyone spends a call on them.

In [4]:
for t in run["targets"]:
    if t["skipped"]:
        print(f"{t['legal_name']:<28} {t['rationale']}")

Firestone                    Chain or franchise (Firestone)
Midas                        Chain or franchise (Midas)
Goodyear                     Chain or franchise (Goodyear)
Crash Champions              Chain or franchise (Crash Champions)
Caliber Collision            Chain or franchise (Caliber Collision)
Fletcher Jones Chicago Service Center Chain or franchise (Fletcher Jones Chicago Service Center)
McGrath Lexus Service Center Chain or franchise (McGrath Lexus Service Center)
Valvoline                    Chain or franchise (Valvoline)
Jiffy Lube                   Chain or franchise (Jiffy Lube)
Rivian Service Center        Chain or franchise (Rivian Service Center)
Midtown Auto Repair          Chain or franchise (Midtown Auto Repair)


## 4. Tier breakdown

In [5]:
print(Counter(t["tier"] for t in run["targets"]))

Counter({'solid': 16, 'reject': 11, 'prime': 3})


## 5. Export for outreach

Same filters as the UI: companies scoring 58 or higher, no chains. The CSV imports straight into a CRM or dialer.

In [6]:
export = client.get(f"/v1/pipeline/runs/{run['id']}/export", params={"min_score": 58})
rows = list(csv.DictReader(io.StringIO(export.text)))
print(len(rows), "rows | columns:", ", ".join(rows[0].keys()))
for r in rows[:3]:
    print(" ", r["fit_score"], r["company"], r["phone"], r["website"])

19 rows | columns: fit_score, tier, company, phone, email, website, address, city, state, notes, vertical, market
  80.0 Gateway Auto Service +1-773-342-7105 https://gatewayautochicago.net/
  80.0 Grand Auto Center +1-312-226-1500 https://www.napaonline.com/en/autocare/?facilityId=576008
  80.0 Augusta & Paulina Auto Service +1-773-486-9768 https://www.aandpautorepair.com/


## 6. Validation

Pydantic rejects bad input before any work happens, and a market the geocoder can't find gets a clear error rather than made-up results.

In [7]:
bad_band = client.post(
    "/v1/pipeline/runs",
    json={"vertical": "Auto", "market": "Chicago, IL", "headcount_min": 50, "headcount_max": 10},
)
print(bad_band.status_code, bad_band.json()["detail"][0]["msg"])

unknown = client.post("/v1/pipeline/runs", json={"vertical": "Auto", "market": "Nowhereville, ZZ"})
print(unknown.status_code, unknown.json()["detail"])

422 headcount_min cannot exceed headcount_max
422 Couldn't find the market 'Nowhereville, ZZ'. Try 'City, ST'.
